## Which models does this notebook use?

**Only the LLM:** `Qwen/Qwen3.5-9B`, split over two GPUs in section 2. This notebook loads no embedding model and no cross-encoder. It reads the retrieval results saved by `rag_retrieval_25q.ipynb` (`outputs_25q/retrieval.json`), so every LLM sees exactly the same contexts.

The retrieval models are defined in `rag_retrieval_25q.ipynb`:
* bi-encoders: `BAAI/bge-small-en-v1.5` and `google/embeddinggemma-2` (float32), in the `EMBEDDERS` dictionary in section 3;
* cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`, `ce = CrossEncoder(...)` in the same section.

The retrieved chunks, in readable form, are in `outputs_25q/retrieved_chunks.md`; the SRD PDF is in `data/SRD_CC_v5.1.pdf`.

# Generation with a local Qwen3.5-9B split across two GPUs

Reads the retrieved contexts saved by `rag_retrieval_25q.ipynb` (same chunks, same top-3 for both embedders and both
pipelines), builds the **same prompt** as the main notebook, and asks `Qwen/Qwen3.5-9B` for an answer.

* **Pipeline 1 (baseline):** first 3 chunks from the embedder. **Pipeline 2:** first 3 chunks after cross-encoder re-ranking.
* **Model placement:** the 9B model is ~18 GB in fp16, more than one 11 GB RTX 2080 Ti, so layers are split over two GPUs with `device_map="auto"`.
* **Thinking off** is run for every question and both embedders. **Thinking on** is run afterwards, `bge-small` first, within a time budget, because it is much slower.
* Identical prompts are generated once and reused (for example when baseline and re-ranked top-3 are the same).

In [1]:
import os, re, json, time, gc, textwrap
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

OUT = "/home2/akshat.pandey/rag/outputs_25q"
LLM_NAME = os.environ.get("MODEL_DIR", "Qwen/Qwen3.5-9B")   # a local copy of Qwen/Qwen3.5-9B on the node's /scratch
BUDGET_SECONDS = 110 * 60          # stop starting new generations after this much generation time
print("gpus:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

R = json.load(open(f"{OUT}/retrieval.json"))
retrieval, chunks = R["retrieval"], R["chunks"]
EMBS = list(retrieval)
QUERIES = list(retrieval[EMBS[0]])
print(len(QUERIES), "questions x", len(EMBS), "embedders")

gpus: ['NVIDIA GeForce RTX 2080 Ti', 'NVIDIA GeForce RTX 2080 Ti']
25 questions x 2 embedders


## 1. Prompt (identical to the main notebook)

In [2]:
PROMPT = """You are a rules assistant for Dungeons & Dragons 5th edition.
Answer the question using ONLY the context passages below. If the passages do not contain the answer, say
"The provided context does not answer this." Be concise (at most 4 sentences) and state any exception or restriction that applies.

Context:
{context}

Question: {question}
Answer:"""

def build_prompt(query, ids):
    ctx = "\n\n".join(f"[Passage {n+1}, page {chunks[i]['page']}] {chunks[i]['text']}" for n, i in enumerate(ids))
    return PROMPT.format(context=ctx, question=query)

## 2. Load the model across both GPUs

In [3]:
tok = AutoTokenizer.from_pretrained(LLM_NAME)
t0 = time.perf_counter()
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="auto",
                                           max_memory={0: "10GiB", 1: "10GiB"})
llm.eval()
print(f"loaded in {time.perf_counter()-t0:.0f}s | class {type(llm).__name__}")
for i in range(torch.cuda.device_count()):
    print(f"gpu{i}: {torch.cuda.memory_allocated(i)/1e9:.1f} GB allocated of {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}")

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

loaded in 14s | class Qwen3_5ForCausalLM
gpu0: 8.5 GB allocated of 11.3
gpu1: 9.4 GB allocated of 11.3


## 3. Generation function

Thinking off: greedy decoding. Thinking on: sampling with a fixed seed (Qwen's recommended temperature 0.6 / top-p 0.95 / top-k 20),
and the text inside `<think>...</think>` is stored separately. If the thinking block never closes, the answer is empty and flagged.

In [4]:
gen_cache = {}
def generate(prompt, thinking):
    key = (prompt, thinking)
    if key in gen_cache:
        return dict(gen_cache[key], cached=True)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=thinking)
    inputs = tok([text], return_tensors="pt").to("cuda:0")
    kw = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20, max_new_tokens=2500) if thinking \
         else dict(do_sample=False, max_new_tokens=250)
    torch.manual_seed(0)
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)
    t = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(**inputs, **kw)
    for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)
    secs = time.perf_counter() - t
    new = out[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(new, skip_special_tokens=True)
    if "</think>" in raw:
        think, answer = raw.split("</think>", 1); truncated = False
    elif not thinking:
        think, answer, truncated = "", raw, False
    else:
        think, answer, truncated = raw, "", True
    res = dict(answer=answer.strip(), think=think.replace("<think>", "").strip(), new_tokens=int(len(new)),
               seconds=secs, truncated=truncated, prompt_tokens=int(inputs["input_ids"].shape[1]))
    gen_cache[key] = res
    return dict(res, cached=False)

# sanity check: the fp16 two-GPU model must produce sensible text before the real run
chk = generate("Reply with just the number: what is 12 plus 30?", False)
print(repr(chk["answer"]), f"{chk['seconds']:.1f}s, {chk['new_tokens']} tokens")
assert "42" in chk["answer"], "model output looks broken (fp16 problem?)"

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


'42' 2.3s, 5 tokens


## 4. Run: all questions, both embedders, both pipelines

Thinking off runs for every question and both embedders (about 10 minutes in total). Results are written to disk after every generation.

**Thinking on is disabled in this run (`THINK_ON` not set).** A first run with it enabled generated at about 12 tokens/s and
15 of the 16 attempts used the whole 2,500-token allowance without finishing their reasoning (mean 212 s each), so almost no
answers came out. Finishing them would need 8,000+ tokens per answer, roughly 11 minutes each, or about 9 hours for the 100
prompts on two RTX 2080 Ti cards. The evidence from that run is kept in `generations_with_thinking_partial.json`.

In [5]:
results, started = [], time.perf_counter()
def save():
    json.dump(dict(results=results, llm=LLM_NAME, budget_seconds=BUDGET_SECONDS,
                   elapsed=time.perf_counter() - started), open(f"{OUT}/generations_qwen35_9b.json", "w"), indent=1)

plan = [(e, False) for e in EMBS]
if os.environ.get("THINK_ON") == "1":          # off by default: see the note in section 4
    plan += [(EMBS[0], True)] + [(e, True) for e in EMBS[1:]]
stopped = False
for emb, thinking in plan:
    for q in QUERIES:
        r = retrieval[emb][q]
        for pipeline, ids in (("baseline", r["base_ids"][:3]), ("rerank", r["rerank_ids"][:3])):
            gold_in_ctx = any(g in ids for g in r["gold_ids"])
            if time.perf_counter() - started > BUDGET_SECONDS:
                stopped = True
                results.append(dict(embedder=emb, query=q, pipeline=pipeline, thinking=thinking, context_ids=ids,
                                    gold_in_context=gold_in_ctx, answer=None, skipped_budget=True))
                continue
            g = generate(build_prompt(q, ids), thinking)
            results.append(dict(embedder=emb, category=r["category"], query=q, pipeline=pipeline, thinking=thinking,
                                context_ids=ids, gold_in_context=gold_in_ctx, **g))
        save()
    print(f"done: embedder={emb} thinking={thinking} | elapsed {(time.perf_counter()-started)/60:.1f} min", flush=True)
print("budget hit, some thinking runs skipped" if stopped else "all generations finished")

done: embedder=bge-small thinking=False | elapsed 7.2 min


done: embedder=embeddinggemma-2 thinking=False | elapsed 13.0 min


all generations finished


## 5. Speed

In [6]:
df = pd.DataFrame([x for x in results if x.get("answer") is not None and not x.get("cached")])
sp = df.groupby("thinking").agg(generations=("seconds", "size"), mean_seconds=("seconds", "mean"),
                                mean_new_tokens=("new_tokens", "mean"), truncated=("truncated", "sum")).round(2)
sp.to_csv(f"{OUT}/qwen35_9b_speed.csv"); sp

,generations,mean_seconds,mean_new_tokens,truncated
thinking,,,,
False,92,8.49,72.41,0


## 6. How often is the gold passage in the context, and what did the model answer?

Answers are graded by hand afterwards (see the summary file); here only the context check is automatic.

In [7]:
dd = pd.DataFrame([x for x in results if x.get("answer") is not None])
ctx = dd.groupby(["embedder", "pipeline"]).gold_in_context.agg(["sum", "count"]).rename(columns={"sum": "gold in top-3", "count": "answers"})
ctx

gold in top-3  answers
embedder         pipeline                        
bge-small        baseline             18       25
                 rerank               21       25
embeddinggemma-2 baseline             16       25
                 rerank               21       25

In [8]:
for emb in EMBS:
    for thinking in (False, True):
        sub = dd[(dd.embedder == emb) & (dd.thinking == thinking)]
        if sub.empty: continue
        print("#" * 100, f"\n{emb} | thinking={'on' if thinking else 'off'}")
        for q in QUERIES:
            for p in ("baseline", "rerank"):
                x = sub[(sub["query"] == q) & (sub.pipeline == p)]
                if x.empty: continue
                x = x.iloc[0]
                print(f"[{p:8s}] ctx_has_gold={x.gold_in_context!s:5s} {x['query'][:60]}\n    ", textwrap.fill(x.answer or "[no answer]", 120, subsequent_indent="     "))

#################################################################################################### 
bge-small | thinking=off
[baseline] ctx_has_gold=False What breaks my focus on a spell?
     Taking damage while concentrating on a spell breaks your focus unless you succeed on a Constitution saving throw.
     Additionally, casting another spell that requires concentration immediately ends your current concentration. You
     can also voluntarily end concentration at any time without using an action. Normal activities, such as moving or
     attacking, do not interfere with concentration.
[rerank  ] ctx_has_gold=True  What breaks my focus on a spell?
     You lose concentration on a spell if you cast another spell that requires concentration, become incapacitated, or die.
     Additionally, taking damage forces you to make a Constitution saving throw (DC 10 or half the damage, whichever is
     higher); failing this breaks your concentration. Normal activity, such as moving or attack